In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
FOLDERNAME = 'cs231n/project/'
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

In [ ]:
# Check GPU availability
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

# Check TensorFlow and PyTorch versions
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("TensorFlow version: ", tf.__version__)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Clone and install dependencies
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile
%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn
!sudo apt-get update
!sudo apt-get install -y colmap
!sudo apt-get install xvfb

In [2]:
project_path = '/content/drive/My Drive/cs231n/project/'
frames_path = project_path + 'frames/'
output_base_path = project_path + 'output/'

In [ ]:
import os

# Prepare Data for Scene 1
scene_num = 1
scene_folder = f'scene{scene_num}'
source_path = os.path.join(frames_path, scene_folder)
output_path = os.path.join(output_base_path, scene_folder)


# Ensure the paths are correctly set
print(f"Source path: {source_path}")
print(f"Output path: {output_path}")

# Verify contents of the input path
input_path = os.path.join(source_path, 'input')
if os.path.exists(input_path) and len(os.listdir(input_path)) > 0:
    print(f"Input directory exists and contains images: {os.listdir(input_path)[:5]}")
else:
    raise Exception(f"Input directory {input_path} does not exist or contains no images.")

# Define paths
database_path = os.path.join(source_path, 'distorted/database.db')
sparse_path = os.path.join(source_path, 'distorted/sparse')
undistorted_path = os.path.join(source_path, 'undistorted')

# Run COLMAP steps with GPU

# Feature extraction
feature_extractor_cmd = f"xvfb-run -a colmap feature_extractor --database_path '{database_path}' --image_path '{input_path}' --ImageReader.single_camera 1 --ImageReader.camera_model OPENCV --SiftExtraction.use_gpu 1 --SiftExtraction.num_threads 24"
print(f"Running feature extraction command: {feature_extractor_cmd}")
!{feature_extractor_cmd}

# Feature matching
feature_matching_cmd = f"xvfb-run -a colmap exhaustive_matcher --database_path '{database_path}' --SiftMatching.use_gpu 1"
print(f"Running feature matching command: {feature_matching_cmd}")
!{feature_matching_cmd}

gpu_usage_cmd = "nvidia-smi"
print("Checking GPU usage:")
!{gpu_usage_cmd}

# Bundle adjustment
mapper_cmd = f"xvfb-run -a colmap mapper --database_path '{database_path}' --image_path '{input_path}' --output_path '{sparse_path}' --Mapper.ba_global_function_tolerance=0.000001"
print(f"Running mapper command: {mapper_cmd}")
!{mapper_cmd}

# Image undistortion
image_undistorter_cmd = f"xvfb-run -a colmap image_undistorter --image_path '{input_path}' --input_path '{sparse_path}/0' --output_path '{undistorted_path}' --output_type COLMAP"
print(f"Running image undistortion command: {image_undistorter_cmd}")
!{image_undistorter_cmd}


In [13]:
import shutil

# Zip the undistorted directory
zip_filename = os.path.join(output_base_path, f'scene{scene_num}.zip')
print(f"Zipping directory {undistorted_path} to {zip_filename}")
shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', undistorted_path)
print(f"Zipped directory {undistorted_path} to {zip_filename}")

Zipping directory /content/drive/My Drive/cs231n/project/frames/scene1/undistorted to /content/drive/My Drive/cs231n/project/output/scene1.zip
Zipped directory /content/drive/My Drive/cs231n/project/frames/scene1/undistorted to /content/drive/My Drive/cs231n/project/output/scene1.zip


In [ ]:
train_script_path = '/content/gaussian-splatting/train.py'
train_cmd = f"python {train_script_path} --data_path '{zip_filename}' --output_path '{output_path}' --use_gpu 1"
print(f"Running training command: {train_cmd}")
!{train_cmd}

In [ ]:
render_script_path = '/content/gaussian-splatting/render.py'
render_cmd = f"python {render_script_path} --model_path '{output_path}' --output_path '{output_path}'"
print(f"Running rendering command: {render_cmd}")
!{render_cmd}

In [ ]:
!pip install pyngrok
from pyngrok import ngrok

ngrok.set_auth_token("2gWHu0eX53vyboV6uhJrDSIAGWR_6dcTE3DgBLkm3ZLz73svy")
public_url = ngrok.connect(port='7860')
print(f"ngrok tunnel: {public_url}")

viewer_cmd = f"python /content/gaussian-splatting/viewer.py --port 7860 --model_path '{output_path}'"
print(f"Running viewer command: {viewer_cmd}")
!{viewer_cmd}